Конфигурация станций и кодов пропусков NOAA

In [1]:
import glob
import os
import numpy as np
import pandas as pd

import sys

sys.path.append('..')  # позволяет импортировать из папки src/, находясь в notebooks/

from src.buoy_config import BUOYS_META, NA_BY_COL

Парсинг сырых файлов, очистка и часовая агрегация (../data/interim/)

In [2]:
os.makedirs('../data/interim', exist_ok=True)
all_hourly_list = []

for st_id, meta in BUOYS_META.items():
  # Убедись, что путь к station_* соответствует твоей локальной папке data/raw/
  files = sorted(glob.glob(f'../data/raw/station_{st_id}/202*/*.txt'))
  dfs = [
      pd.read_csv(f, sep=r'\s+', skiprows=lambda x: x == 1, na_values=NA_BY_COL)
      for f in files
  ]
  raw = pd.concat(dfs, ignore_index=True)

  # Собираем timestamp и округляем до начала часа
  raw['timestamp'] = pd.to_datetime(
      raw[['#YY', 'MM', 'DD', 'hh', 'mm']].rename(
          columns={
              '#YY': 'year',
              'MM': 'month',
              'DD': 'day',
              'hh': 'hour',
              'mm': 'minute',
          }
      )
  )
  raw['timestamp_hour'] = raw['timestamp'].dt.floor('h')

  # Агрегируем в 1 час
  hourly = (
      raw.sort_values('timestamp')
      .groupby('timestamp_hour', as_index=False)
      .agg(
          WDIR=('WDIR', 'last'),
          WSPD_mean=('WSPD', 'mean'),
          WSPD_std=('WSPD', 'std'),
          GST_max=('GST', 'max'),
          WVHT=('WVHT', 'last'),
          DPD=('DPD', 'last'),
          APD=('APD', 'last'),
          MWD=('MWD', 'last'),
          PRES=('PRES', 'mean'),
          ATMP=('ATMP', 'mean'),
          WTMP=('WTMP', 'mean'),
          DEWP=('DEWP', 'mean'),
      )
      .rename(columns={'timestamp_hour': 'timestamp'})
  )

  # Выравниваем на непрерывную часовую сетку
  grid = pd.DataFrame({
      'timestamp': pd.date_range(
          hourly['timestamp'].min(), hourly['timestamp'].max(), freq='1h'
      )
  })
  hourly = grid.merge(hourly, on='timestamp', how='left')

  # Добавляем физические метаданные буя
  hourly['station_id'] = st_id
  hourly['lat'] = meta['lat']
  hourly['lon'] = meta['lon']
  hourly['depth_m'] = meta['depth_m']

  first_cols = ['timestamp', 'station_id', 'lat', 'lon', 'depth_m']
  other_cols = [c for c in hourly.columns if c not in first_cols]
  hourly = hourly[first_cols + other_cols]

  # Сохраняем отдельный часовой CSV для каждого буя
  out_path = f'../data/interim/buoy_{st_id}_hourly.csv'
  hourly.to_csv(out_path, index=False)
  all_hourly_list.append(hourly)

  valid_wvht = hourly['WVHT'].notna().sum()
  print(
      f"[OK] Буй {st_id} ({meta['name']}): всего часов = {len(hourly)}, с"
      f' известной волной WVHT = {valid_wvht}'
  )

all_buoys_hourly = pd.concat(all_hourly_list, ignore_index=True)
all_buoys_hourly.to_csv('../data/interim/all_buoys_hourly.csv', index=False)

print('\nИтоговый размер общей часовой таблицы:', all_buoys_hourly.shape)
print(
    'Всего чистых часов с известной волной WVHT по всем 6 буям:',
    all_buoys_hourly['WVHT'].notna().sum(),
)

[OK] Буй 46059 (West California): всего часов = 43824, с известной волной WVHT = 30124
[OK] Буй 46022 (Eel River): всего часов = 43824, с известной волной WVHT = 25298
[OK] Буй 46014 (Point Arena): всего часов = 43824, с известной волной WVHT = 43473
[OK] Буй 46013 (Bodega Bay): всего часов = 43824, с известной волной WVHT = 28676
[OK] Буй 46026 (San Francisco): всего часов = 43824, с известной волной WVHT = 40796
[OK] Буй 46028 (Cape San Martin): всего часов = 43824, с известной волной WVHT = 42697

Итоговый размер общей часовой таблицы: (262944, 17)
Всего чистых часов с известной волной WVHT по всем 6 буям: 211064


Формирование целевых переменных (+3ч, +6ч) и хронологический сплит (../data/processed/)

In [3]:
# 1. Формируем целевые переменные (+3ч и +6ч вперёд) строго внутри каждого буя
df_model = all_buoys_hourly.sort_values(['station_id', 'timestamp']).copy()
df_model['target_wvht_3h'] = df_model.groupby('station_id')['WVHT'].shift(-3)
df_model['target_wvht_6h'] = df_model.groupby('station_id')['WVHT'].shift(-6)

# 2. Оставляем строки с известным таргетом
df_model = df_model.dropna(
    subset=['WVHT', 'target_wvht_3h', 'target_wvht_6h']
).reset_index(drop=True)

# 3. Разделяем строго по времени: 2021–2024 (train) и 2025 (test)
train_df = df_model[df_model['timestamp'].dt.year < 2025].reset_index(drop=True)
test_df = df_model[df_model['timestamp'].dt.year == 2025].reset_index(drop=True)

os.makedirs('../data/processed', exist_ok=True)
train_df.to_csv('../data/processed/train.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print(
    f'train.csv (2021–2024): {train_df.shape}'
    f' ({len(train_df)/len(df_model):.1%})'
)
print(
    f'test.csv  (2025):      {test_df.shape}'
    f' ({len(test_df)/len(df_model):.1%})'
)

train.csv (2021–2024): (168873, 19) (81.2%)
test.csv  (2025):      (39157, 19) (18.8%)
